https://youtu.be/W-4ujGEHR1o

In [0]:
king_data = [
    (1, 'Robb Stark', 'House Stark'),
    (2, 'Joffrey Baratheon', 'House Lannister'),
    (3, 'Stannis Baratheon', 'House Baratheon'),
    (4, 'Balon Greyjoy', 'House Greyjoy'),
    (5, 'Mace Tyrell', 'House Tyrell'),
    (6, 'Doran Martell', 'House Martell')
]

battle_data = [
    (1, 'Battle of Oxcross', 1, 2, 1, 'The North'),
    (2, 'Battle of Blackwater', 3, 4, 0, 'The North'),
    (3, 'Battle of the Fords', 1, 5, 1, 'The Reach'),
    (4, 'Battle of the Green Fork', 2, 6, 0, 'The Reach'),
    (5, 'Battle of the Ruby Ford', 1, 3, 1, 'The Riverlands'),
    (6, 'Battle of the Golden Tooth', 2, 1, 0, 'The North'),
    (7, 'Battle of Riverrun', 3, 4, 1, 'The Riverlands'),
    (8, 'Battle of Riverrun', 1, 3, 0, 'The Riverlands')
]


king_schema = "k_no int , king string , house string"
battle_schema = "battle_number int , name string ,attacker_king int , defender_king int , attacker_outcome int , region string"

king_df = spark.createDataFrame(data = king_data ,schema = king_schema)
battle_df = spark.createDataFrame(data = battle_data ,schema = battle_schema)

king_df.display()
battle_df.display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [0]:
battle_df_result = (
    battle_df
        .withColumn("winner", F.when(F.col("attacker_outcome")==1 ,F.col("attacker_king")).otherwise(F.col("defender_king")))
        .orderBy("winner")
        #.join(king_df, "battle_df.winner == king_df.k_no" ,"left")
        .groupBy("region","winner").agg(F.count("winner").alias("count_winner"))
        .withColumn("rank", F.dense_rank().over(Window.partitionBy("region").orderBy(F.col("count_winner").desc())))
        .filter("rank == 1")
)

battle_df_result.display()



In [0]:
result = (
    battle_df_result
    .join(king_df, battle_df_result.winner == king_df.k_no,"left")
        #.join(king_df, "battle_df_result.winner == king_df.k_no", "left")
        .select("region","house","count_winner")
)


result.display()